# 🐙 Octopus Cloud

Free cloud GPU for **training Octopus brains**. Octopus makes 3D models and images in the free cloud by itself, so the 3D and image cells here are a backup for when its daily free allowance runs out.

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save.

Then run the cells you need from top to bottom (click ▶ on each one). Every result downloads to your computer, and you import it into Octopus:

- Brains: **Train Octopus → Import brain** (the `-brain.zip` file)
- 3D models: **Modelling → Import a 3D model** (the `.zip` file)

Free Colab sessions stop after a few hours or when idle, so download results as soon as they finish.

In [ ]:
#@title 1. Set up Octopus (run this first, about 1 minute)
import os, subprocess, sys
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
if not gpu:
    raise SystemExit("No GPU. Go to Runtime → Change runtime type → T4 GPU, then run this cell again.")
print("GPU:", gpu)
if not os.path.exists("/content/octopus"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/physicstheory/octopus.git", "/content/octopus"], check=True)
os.environ["OCTOPUS_HOME"] = "/content/octopus_home"
sys.path.insert(0, "/content/octopus")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft>=0.17", "accelerate", "bitsandbytes>=0.45", "datasets", "trimesh>=4.4", "diffusers", "xatlas", "fast-simplification", "scipy"], check=True)
print("✅ Octopus is ready")

## 🧠 Train a brain

Upload your own examples (the file from **Train Octopus → Download my data**, or any `.jsonl`, `.csv` or `.txt`) and/or tick open datasets.

Rough times on a free T4 for about 2,000 examples:
- **Hatchling, Reef, Abyss** (0.5B to 3B): 15 to 40 minutes.
- **Cuttlefish, Nautilus, Coral** (about 4B): 40 to 80 minutes.
- **Kraken, Trench, Leviathan, Orca** (7B to 8B): 1.5 to 3 hours. These train in 4-bit to fit the free GPU.
- **Manta, Megalodon** (12B to 14B): fit, but can run into the free session's time limit. Use fewer examples (500) and Quick.

Free Colab can stop a session after a few hours, so start with a smaller model or fewer examples if you're unsure.

In [ ]:
#@title 2. Choose what to train on
BRAIN_NAME = "octopus-v1" #@param {type:"string"}
BASE_MODEL = "Reef (1.5B, Qwen 2.5 Coder)" #@param ["Hatchling (0.5B, Qwen 2.5 Coder)", "Reef (1.5B, Qwen 2.5 Coder)", "Abyss (3B, Qwen 2.5 Coder)", "Cuttlefish (3.8B, Microsoft Phi-4 mini)", "Nautilus (4B, Qwen 3)", "Coral (4B, Google Gemma 3)", "Kraken (7B, Qwen 2.5 Coder)", "Trench (7B, DeepSeek R1)", "Leviathan (8B, Qwen 3)", "Orca (8B, Meta Llama 3.1)", "Manta (12B, Google Gemma 3)", "Megalodon (14B, Qwen 2.5 Coder)"]
UPLOAD_MY_DATA = True #@param {type:"boolean"}
HELPFUL_CHAT_NoRobots = True #@param {type:"boolean"}
GENERAL_KNOWLEDGE_Dolly = False #@param {type:"boolean"}
CODING_CodeAlpaca = True #@param {type:"boolean"}
PYTHON_CODING_18k = False #@param {type:"boolean"}
EXAMPLES_PER_OPEN_DATASET = 1500 #@param {type:"integer"}
HOW_HARD = "Balanced" #@param ["Quick", "Balanced", "Thorough"]

from octopus import data
from octopus.config import BASE_MODELS
BASE_ID = next(k for k, v in BASE_MODELS.items() if BASE_MODEL.startswith(v["label"] + " "))
print("Base model:", BASE_ID)
chosen = []
if UPLOAD_MY_DATA:
    from google.colab import files
    print("Choose your file(s), or press Cancel to skip")
    try:
        up = files.upload()
    except Exception:
        up = {}
    for name, raw in up.items():
        ds = data.save_upload(name, raw.decode("utf-8", errors="replace"))
        chosen.append(ds["id"]); print(f"Added {ds['count']} examples from {name}")
for flag, key in [(HELPFUL_CHAT_NoRobots, "no_robots"), (GENERAL_KNOWLEDGE_Dolly, "dolly"),
                  (CODING_CodeAlpaca, "code_alpaca"), (PYTHON_CODING_18k, "python_18k")]:
    if flag:
        chosen.append("builtin:" + key)
if not chosen:
    raise SystemExit("Pick at least one dataset or upload a file.")
print("Training on:", chosen)

In [ ]:
#@title 3. Train (watch the loss go down)
import time
from octopus.engine import Engine
from octopus.trainer import Trainer, TrainConfig

epochs, rank = {"Quick": (1, 8), "Balanced": (2, 16), "Thorough": (3, 32)}[HOW_HARD]
trainer = Trainer(Engine())
trainer.start(TrainConfig(name=BRAIN_NAME, base=BASE_ID, datasets=chosen, epochs=epochs, lora_rank=rank,
                          batch_size=4, grad_accum=4, builtin_limit=EXAMPLES_PER_OPEN_DATASET))
seq, final = 0, None
while trainer.running or seq < len(trainer.events):
    for ev in trainer.wait_events(seq, 5):
        seq = ev["seq"] + 1
        if ev["t"] == "log":
            print(ev["msg"])
        elif ev["t"] == "status" and ev["status"] in ("done", "error", "stopped"):
            final = ev
if not final or final["status"] == "error":
    raise SystemExit("Training failed: " + str(final and final.get("error")))
BRAIN_DIR = f"/content/octopus_home/brains/{final['brain']}"
print("\n✅ Trained! Test answer:\n", final.get("sample", ""))

In [ ]:
#@title 4. Package the brain for Octopus and download it (5 to 30 minutes, bigger models take longer)
#@markdown Makes one `.zip` that works on any computer: a `.gguf` model for Ollama, plus the small adapter for NVIDIA setups.
#@markdown Big brains are several GB, so saving to Google Drive is more reliable than a browser download.
SAVE_TO_GOOGLE_DRIVE = False #@param {type:"boolean"}
import gc, glob, os, shutil, subprocess, sys, zipfile, torch
from octopus.merge import merge

gc.collect(); torch.cuda.empty_cache()
name = os.path.basename(BRAIN_DIR)
merged = "/content/merged"
shutil.rmtree(merged, ignore_errors=True)
merge(BASE_ID, BRAIN_DIR, merged)  # one file at a time, so even 8B to 14B models fit in Colab's memory

if not os.path.exists("/content/llama.cpp"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp", "/content/llama.cpp"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "/content/llama.cpp/gguf-py", "sentencepiece", "mistral-common"], check=True)
billions = float(BASE_MODELS[BASE_ID]["size"].rstrip("B"))
gguf = f"/content/{name}.gguf"
if billions < 3:
    subprocess.run([sys.executable, "/content/llama.cpp/convert_hf_to_gguf.py", merged, "--outfile", gguf, "--outtype", "q8_0"], check=True)
else:
    # Bigger models are shrunk to 4-bit (Q4_K_M), the same size Ollama uses for its own copies.
    if not os.path.exists("/content/llama.cpp/build/bin/llama-quantize"):
        print("Building the quantizer (about 3 minutes, once per session)…")
        subprocess.run(["cmake", "-S", "/content/llama.cpp", "-B", "/content/llama.cpp/build", "-DGGML_CUDA=OFF", "-DLLAMA_CURL=OFF"], check=True, capture_output=True)
        subprocess.run(["cmake", "--build", "/content/llama.cpp/build", "--target", "llama-quantize", "-j", str(os.cpu_count() or 2)], check=True, capture_output=True)
    f16 = f"/content/{name}-f16.gguf"
    subprocess.run([sys.executable, "/content/llama.cpp/convert_hf_to_gguf.py", merged, "--outfile", f16, "--outtype", "f16"], check=True)
    shutil.rmtree(merged, ignore_errors=True)
    subprocess.run(["/content/llama.cpp/build/bin/llama-quantize", f16, gguf, "Q4_K_M"], check=True)
    os.remove(f16)
shutil.rmtree(merged, ignore_errors=True)

pack = f"/content/{name}-brain.zip"
with zipfile.ZipFile(pack, "w", zipfile.ZIP_STORED) as z:  # the model is already compressed
    z.write(gguf, f"{name}/{name}.gguf")
    for f in glob.glob(f"{BRAIN_DIR}/*"):
        z.write(f, f"{name}/{os.path.basename(f)}")
os.remove(gguf)
size = os.path.getsize(pack) / 1e9
if SAVE_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/Octopus", exist_ok=True)
    shutil.copy(pack, "/content/drive/MyDrive/Octopus/")
    print(f"✅ Saved {os.path.basename(pack)} ({size:.1f} GB) to Google Drive → Octopus. Download it from drive.google.com, then in Octopus: Train Octopus → Import brain.")
else:
    print(f"✅ {os.path.basename(pack)} ({size:.1f} GB). Import it in Octopus: Train Octopus → Import brain.")
    from google.colab import files
    files.download(pack)

## 🧊 3D models (Hunyuan3D-2)

Text or picture → game-ready 3D model. Octopus makes the shape and texture with Hunyuan3D, then finishes it like a game artist would: cleans the mesh, reduces it to your triangle budget, unwraps UVs, bakes PBR colour, roughness/metal and normal maps, and builds LOD0 to LOD2. You get a game pack with GLB, OBJ and STL.

You usually don't need this: Octopus makes 3D models in the free cloud by itself. Use the notebook when the free daily cloud allowance runs out. The first run installs Hunyuan3D and builds its texture renderer, which takes about 10 minutes.

Hunyuan3D-2 is released under the Tencent Hunyuan 3D 2.0 Community License: free to use, but not in the EU, UK or South Korea, and products with over 1 million monthly users need a licence from Tencent.

In [ ]:
#@title 5. Install Hunyuan3D (once per session, about 10 minutes)
import os, subprocess, sys
if not os.path.exists("/content/Hunyuan3D-2"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Tencent/Hunyuan3D-2", "/content/Hunyuan3D-2"], check=True)
os.chdir("/content/Hunyuan3D-2")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
for ext in ["hy3dgen/texgen/custom_rasterizer", "hy3dgen/texgen/differentiable_renderer"]:
    print("Building", ext, "…")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "."], cwd=ext, capture_output=True, text=True)
    print("  ok" if r.returncode == 0 else "  failed, textures will be skipped:\n" + r.stderr[-1500:])
os.chdir("/content")
import site, importlib; importlib.invalidate_caches(); sys.path.insert(0, "/content/Hunyuan3D-2")
print("✅ Hunyuan3D installed")

In [ ]:
#@title 6. Make a 3D model
PROMPT = "A wooden treasure chest with iron bands and a gold lock" #@param {type:"string"}
USE_MY_PICTURE_INSTEAD = False #@param {type:"boolean"}
QUALITY = "quality" #@param ["fast", "quality"]
TEXTURES = True #@param {type:"boolean"}
TRIANGLE_BUDGET = "game" #@param ["mobile", "game", "hero", "cinematic"]
TEXTURE_SIZE = 2048 #@param [1024, 2048, 4096] {type:"raw"}
HEIGHT_IN_METRES = 1.0 #@param {type:"number"}
MAKE_LODS = True #@param {type:"boolean"}

import base64, os, shutil
from IPython.display import HTML, display
from octopus import meshkit, studio as st

studio = st.Studio(free_others=lambda: None)
finish = meshkit.FinishOptions.from_request({"budget": TRIANGLE_BUDGET, "texture_size": TEXTURE_SIZE,
                                             "height": HEIGHT_IN_METRES, "lods": MAKE_LODS})
image_id = None
if USE_MY_PICTURE_INSTEAD:
    from google.colab import files
    up = files.upload()
    name, raw = next(iter(up.items()))
    image_id = st.save_upload(name, "data:image/png;base64," + base64.b64encode(raw).decode())["id"]

asset = None
for ev in studio.make_model(PROMPT, image_id=image_id, preset=QUALITY, texture=TEXTURES, finish=finish, where="local"):
    if ev["t"] == "progress":
        print(f"{(ev.get('pct') or 0):5.0f}%  {ev['stage']}")
    elif ev["t"] == "error":
        raise SystemExit("❌ " + ev["d"])
    elif ev["t"] == "done":
        asset = ev["asset"]

folder = f"/content/octopus_home/assets/{asset['id']}"
glb = open(f"{folder}/{asset['files']['glb']}", "rb").read()
display(HTML(f'''<script type="module" src="https://cdn.jsdelivr.net/npm/@google/model-viewer@3.5.0/dist/model-viewer.min.js"></script>
<model-viewer src="data:model/gltf-binary;base64,{base64.b64encode(glb).decode()}" camera-controls auto-rotate shadow-intensity="1"
 style="width:100%;height:420px;background:#111;border-radius:16px"></model-viewer>'''))
safe = "".join(c if c.isalnum() else "-" for c in (PROMPT or "model"))[:40].strip("-") or "model"
zip_path = shutil.make_archive(f"/content/{safe}", "zip", folder)
stats = asset.get("stats") or {}
print(f"✅ {stats.get('faces', 0):,} triangles, {stats.get('texture')}px PBR textures, {len(stats.get('lods', []))} LODs. Downloading {os.path.basename(zip_path)}. Import it in Octopus: Modelling → Import a 3D model.")
from google.colab import files
files.download(zip_path)

## 🖼️ Images

In [ ]:
#@title 7. Make images
PROMPT = "A floating island city at sunset, fantasy concept art" #@param {type:"string"}
MODEL = "sdxl" #@param ["sdxl", "flux"]
SHAPE = "1:1" #@param ["1:1", "16:9", "9:16", "4:3"]
HOW_MANY = 2 #@param {type:"integer"}

from IPython.display import Image as Show, display
from octopus import studio as st
from google.colab import files
studio = st.Studio(free_others=lambda: None)
for ev in studio.make_image(PROMPT, MODEL, SHAPE, HOW_MANY):
    if ev["t"] == "error":
        raise SystemExit("❌ " + ev["d"])
    if ev["t"] == "done":
        asset = ev["asset"]
folder = f"/content/octopus_home/assets/{asset['id']}"
for name in asset["images"]:
    display(Show(f"{folder}/{name}", width=420))
    files.download(f"{folder}/{name}")